### 01_manifest

Builds basic manifests for VGGFace2
* Builds single CSV for these two directories:
    * `data_raw/vggface2/train/<person_id>/*.jpg`
    * `data_raw/vggface2/val/<person_id>/*.jpg`
* Out: `data_processed/vggface2/manifests/manifest_basic.csv`



In [ ]:
# ----- Imports and config -----
import sys
from pathlib import Path

import hashlib
from PIL import Image, UnidentifiedImageError
import imagehash
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [17]:
# ===== Uncomment and run only if you need to reload imports =====

# import importlib
# import scripts.config as cfg
# importlib.reload(cfg)

# ===============================================================  

In [18]:
# ----- Helper functions -----

# used for detection of duplicates (i.e. bytewise identical files)
def sha1_of_file(path: Path, block_size: int = 65536):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block_size), b""):
            h.update(b)
    return h.hexdigest()

# used for detection of near-duplicates (i.e. same image resized/recompressed)
def phash_of_image(path: Path):
    try:
        return str(imagehash.average_hash(Image.open(path)))
    except Exception:
        return None

def to_rel(p: Path) -> str:
    """Return a stable relative path string from repo_root, fallback to name."""
    try:
        return str(p.relative_to(repo_root).as_posix())
    except Exception:
        try:
            return str(p.relative_to(Path.cwd()).as_posix())
        except Exception:
            return str(p.name)

# path conversion 
def abs_from_rel(rel_path):
    p = Path(rel_path)
    if p.is_absolute():
        return p
    return (PROJECT_ROOT / p).resolve()

# convert absolute path to relative path from project root
def rel_from_abs(abs_path):
    return Path(abs_path).resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()

In [30]:
# ----- Main function for image manifest builder -----

def manifest_builder(IN_DIR, OUT_DIR, SAMPLE_LIMIT = None):
    """
    Manifest builder (per-dataset + master)
     - Scans ROOT_DIR for media files
     - Computes sha1, perceptual hash (image only), media size, readable flag
     - Writes per-dataset manifest: data_processed/<dataset>/manifests/manifest_basic.csv
     - Writes master manifest: data_processed/manifests/master_manifest_basic.csv

    """
    # img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pgm", ".webp"}
    # vid_exts = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv", ".mpg", ".mpeg"}
    # exts = img_exts | vid_exts
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pgm", ".webp"}

    # create directories if needed
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "manifests").mkdir(parents=True, exist_ok=True)

    if not IN_DIR.exists():
        raise RuntimeError(f"Input data directory {IN_DIR} does not exist; please download and extract the dataset there before proceeding.")

    # collect files
    all_files = []
    count = 0
    for p in IN_DIR.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in exts:
            continue
        all_files.append(p)
        count += 1
        if SAMPLE_LIMIT and count >= SAMPLE_LIMIT:
            break
    print(f"Found {len(all_files)} image files under {to_rel(IN_DIR)}")

    # build manifest master rows 
    master_rows = []
    # build per-dataset lists to write separate CSVs
    per_dataset_rows = {}

    for p in tqdm(sorted(all_files), desc="Scanning media files"):
        try:
            rel = p.relative_to(IN_DIR)
        except Exception:
            # unexpected path; fallback to name only
            rel = Path(p.name)
        
        parts = rel.parts
        dataset = parts[0] if len(parts) > 0 else ""        
        split = parts[1] if len(parts) > 1 else ""
        person_raw = parts[2] if len(parts) > 2 else p.parent.name
        person_id = person_raw
        
        # Ensure per-dataset output directories exist
        ds_out = OUT_DIR / dataset
        (ds_out / "aligned").mkdir(parents=True, exist_ok=True)
        (ds_out / "embeddings").mkdir(parents=True, exist_ok=True)
        (ds_out / "manifests").mkdir(parents=True, exist_ok=True)

        # Read image metadata
        w = h = fmt = None
        readable = False
        try:
            with Image.open(p) as im:
                w, h = im.size
                fmt = im.format
                readable = True
        except (UnidentifiedImageError, OSError, ValueError):
            readable = False
        except Exception:
            readable = False

        # compute perceptual hash for near-duplicate detection (images only)
        phash = None
        try:
            phash = phash_of_image(p)
        except Exception:
            phash = None

        # compute sha1 for duplicate detection
        sha1 = None
        try:
            sha1 = sha1_of_file(p)
        except Exception:
            sha1 = None
        

        row = {
            "image_path": rel_from_abs(p),  # path to original image
            "dataset": dataset,             # source labels (e.g. vggface2,ibeta)
            "split": split,                 # split (e.g. train, val, test)
            "person_id": person_id,         # person id (e.g. n000002)
            "person_raw": person_raw,       # raw person id extracted from file, so person_id can be renamed/reconstructed
            "sha1": sha1,                   # sha1 hash of file (checksum that is used to detect duplicates)
            "phash": phash,                 # perceptual hash of image (for near-duplicate detection)
            "width": w,                     # image width in pixels
            "height": h,                    # image height in pixels
            "format": fmt,                  # image format (e.g. JPEG, PNG)
            "readable": bool(readable)      # whether image could be opened/read
        }

        master_rows.append(row)
        per_dataset_rows.setdefault(dataset, []).append(row)

    # Write master manifest
    master_df = pd.DataFrame(master_rows)
    master_path = OUT_DIR / "manifests" / "master_manifest_basic.csv"
    master_df.to_csv(master_path, index=False)
    print(f"Wrote master manifest: {to_rel(master_path)} rows: {len(master_df)}")

    # Write per-dataset manifests
    for dataset, rows in per_dataset_rows.items():
        ds_manifest_path = OUT_DIR / dataset / "manifests" / "manifest_basic.csv"
        pd.DataFrame(rows).to_csv(ds_manifest_path, index=False)
        print(f"Wrote {dataset} manifest: {to_rel(ds_manifest_path)} rows: {len(rows)}")

    print("Manifest building complete.")

    return master_df


In [32]:
# ----- Build Manifest CSVs -----

# import and define paths
from scripts.config import (
    PROJECT_ROOT,
    DATA_RAW,
    DATA_PROCESSED
)

DS_DATASET = "vggface2"
MANIFEST_BASIC = DATA_PROCESSED / DS_DATASET / "manifests" / "manifest_basic.csv"

if MANIFEST_BASIC.exists():
    print(f"Manifest already exists at {to_rel(MANIFEST_BASIC)}; skipping manifest building.")
    df = pd.read_csv(MANIFEST_BASIC)
else:
    print("Building manifests...")
    df = manifest_builder(DATA_RAW, DATA_PROCESSED, SAMPLE_LIMIT=None)



Found 197693 image files under data_raw


Scanning media files: 100%|██████████| 197693/197693 [43:35<00:00, 75.60it/s] 


Wrote master manifest: master_manifest_basic.csv rows: 197693
Wrote vggface2 manifest: manifest_basic.csv rows: 197693
Manifest building complete.
Manifest already exists at manifest_basic.csv; skipping manifest building.
